In [1]:
!pip install -q \
langchain==0.3.25 \
langchain-core==0.3.59 \
langchain-community==0.3.24 \
langchain-text-splitters==0.3.8 \
langchain-google-genai==2.1.4 \
faiss-cpu \
sentence-transformers \q

In [4]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [5]:
# All imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_core.documents import Document
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown


In [6]:
# Cell 4 — Create text document
text = """
Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.
She completed an Interview Bot project using LangChain and Gemini.
"""

print("Text document created!")
print(f"Total characters in document: {len(text)}")

Text document created!
Total characters in document: 381


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [8]:
# See what each chunk looks like
for i, chunk in enumerate(chunks):
    print(f"\n📦 CHUNK {i+1}:")
    print("-" * 40)
    print(chunk)
    print(f"Characters in this chunk: {len(chunk)}")


📦 CHUNK 1:
----------------------------------------
Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.
Characters in this chunk: 152

📦 CHUNK 2:
----------------------------------------
She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.
Characters in this chunk: 189

📦 CHUNK 3:
----------------------------------------
She completed an Interview Bot project using LangChain and Gemini.
Characters in this chunk: 66


In [16]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = FAISS.from_texts(chunks, embeddings)

print("Embeddings created and stored in FAISS!")
print(f"Total vectors stored: {len(chunks)}")

Embeddings created and stored in FAISS!
Total vectors stored: 3


In [17]:
# Create retriever from vectorstore
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print(" Retriever created!")
print("Retriever will return 2 most relevant chunks per question")

 Retriever created!
Retriever will return 2 most relevant chunks per question


In [20]:
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.0
    ),
    retriever=retriever,
    return_source_documents=True
)
print("RAG chain created!")
print("Pipeline: Question --> Retriever -->FAISS --> Gemini --> Answer")

RAG chain Created
Pipeline: Question --> Retriever -->FAISS --> Gemini --> Answer


In [23]:
def ask(question):
    print(f"\n QUESTION: {question}")
    print("-" * 50)

    # Invoke the RAG chain
    result = qa_chain.invoke({"query": question})

    # Print the answer
    print(" ANSWER:")
    display(Markdown(result["result"]))

    # Print which chunks were used
    print("\n SOURCE CHUNKS USED:")
    for i, doc in enumerate(result["source_documents"]):
        print(f"\nChunk {i+1}: {doc.page_content}")

print("ask() function ready!")

ask() function ready!


In [24]:
ask("Where does Manpreet study?")


 QUESTION: Where does Manpreet study?
--------------------------------------------------
 ANSWER:


Manpreet studies at Clarkson University.


 SOURCE CHUNKS USED:

Chunk 1: Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.

Chunk 2: She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.


In [25]:
ask("How many years of experience does Manpreet have?")


 QUESTION: How many years of experience does Manpreet have?
--------------------------------------------------
 ANSWER:


Manpreet has 3 years of experience.


 SOURCE CHUNKS USED:

Chunk 1: Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.

Chunk 2: She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.


In [26]:
ask("What are Manpreet's skills?")


 QUESTION: What are Manpreet's skills?
--------------------------------------------------
 ANSWER:


Manpreet's skills include Python, Machine Learning, LangChain, and FAISS. She is also learning RAG pipelines and AI engineering.


 SOURCE CHUNKS USED:

Chunk 1: Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.

Chunk 2: She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.


In [27]:
ask("What is Manpreet's favorite food?")


 QUESTION: What is Manpreet's favorite food?
--------------------------------------------------
 ANSWER:


I don't know Manpreet's favorite food. The provided context does not contain this information.


 SOURCE CHUNKS USED:

Chunk 1: Manpreet Kaur is an MS Data Science student at Clarkson University.
She has 3 years of experience as ML Engineer at Wipro.
She is on STEM OPT till 2028.

Chunk 2: She is on STEM OPT till 2028.
She is learning LangChain, RAG pipelines and AI engineering.
She is on a 90 day job search plan.
Her skills include Python, Machine Learning, LangChain, FAISS.
